# 26 — GBSA parameter vs MD

Per-axis Spearman ρ between each MD signature and each GBSA parameter preference (which `igb` / `intdiel` / `saltcon` / `surften` level a target prefers). Simpler than Act 4's ML NBs and more diagnostic: does MD carry any information about GBSA choice?

**Method.** Stars mark uncorrected p < 0.05. The BH-FDR (Benjamini–Hochberg) pass was run separately (see NB 27's fuller correction); nothing survives at q < 0.10. Read stars as a visual guide only.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/26_gbsa_param_vs_md_figK.png`.)_

> **Reader guide.** *Experiment A3 (see [STUDY_DESIGN §A3](../../STUDY_DESIGN.md)):* per-complex
> MD-feature analysis and downstream ranking questions.
>
> See STUDY_DESIGN Chapter §A3 Q1 (per-target combo selection) and Q2 (single-feature panel
> ranker) for the framing this notebook addresses.
>
> **Reproducibility contract:** reads `data/derived/features.parquet` +
> `data/raw/reference/ohds_metadata.csv` (and `data/derived/canonical_baselines.csv` for
> baseline comparison).

In [ ]:
# --- notebook preamble ---
NB_STEM = "40_gbsa_param_vs_md"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 16. Do MD signatures predict which GBSA parameter is best?

The § 12–14 ML models tried the hardest version: predict the best global sp-config (or GBSA combo) from an MD fingerprint under LOTO. With 8–9 targets, that couldn't lift panel BEDROC above the locked baseline.

**Easier and more diagnostic question:** for each GBSA parameter axis on its own, does a target's MD signature predict which level of that axis the target prefers?

- **`igb`** (implicit-solvent generation): 1 vs 8 — Born-radii algorithm.
- **`intdiel`** (internal dielectric): 1 vs 4 — how polarisable the protein interior is.
- **`saltcon`** (salt concentration): 0.0 vs 0.15 M — electrostatic screening length.
- **`surften`** (nonpolar SA term): 0 vs 0.0072 — nonpolar solvation.

For each axis we compute a per-target preference delta: BEDROC α=20 at the high level minus BEDROC at the low level, marginalised over the other three axes. Then Spearman ρ (per-target-delta, per-target-MD-feature) across the 8 targets, permutation p-value.

Dataset: `bedroc_all_combos_per_target.csv` from the temporal analysis — 48 GBSA combos × 9 targets = 432 per-target BEDROC values, the closest we have to a full factorial over the GBSA parameter space.

In [ ]:

from scipy.stats import spearmanr

# NOTE(fix-pack iter1): removed hardcoded absolute path — GBSA_STUDY comes from discovery9.paths
bt = pd.read_csv(f'{GBSA_STUDY}/data/derived/temporal/bedroc_all_combos_per_target.csv')

# per-target MD fingerprint
FP_MD = [
    'rmsd_bb_mean_A','rmsd_as_bb_mean_A','protein_rg_mean_A','as_ca_rmsf_mean_A',
    'lig_drift_mean_A','lig_com_disp_max_A','lig_escape_frac',
    'lig_internal_rmsd_mean_A','lig_rmsf_mean_A',
    'lig_buried_sasa_mean_A2','vdw_contacts_mean',
    'n_hb_mean','hb_persistence_frac','salt_bridges_lp_mean',
    'ifp_tanimoto_median_vs_ref','lig_binding_modes_2A','lig_orient_autocorr_mean',
    'lig_rg_mean_A','lig_asphericity_mean','lig_dipole_mean_eA',
    'active_site_formal_charge','protein_formal_charge','n_active_site_residues',
    'ligand_partial_charge_sum',  # iter-2 FIX 4b: required by scatter panels below
]
FP = df.groupby('target')[FP_MD].mean()
targets = sorted(set(FP.index) & set(bt.target.unique()))
FP = FP.loc[targets]
print(f'{len(targets)} targets · {bt.combo.nunique()} GBSA combos · {len(FP_MD)} MD features')

# Per-target preference deltas
def perm_p(rho, x, y, n=5000, seed=0):
    rng = np.random.default_rng(seed); s = abs(rho); ct = 0
    for _ in range(n):
        yp = rng.permutation(y)
        r, _ = spearmanr(x, yp)
        if abs(r) >= s: ct += 1
    return ct/n

axes = {
    'igb 1→8':     ('igb',     1, 8),
    'intdiel 1→4': ('intdiel', 1, 4),
    'salt 0→0.15': ('saltcon', 0.0, 0.15),
    'surften 0→0.0072': ('surften', 0.0, 0.0072),
}

# build a (feature × axis) heatmap of Spearman ρ + p
rho_mat  = pd.DataFrame(index=FP_MD, columns=list(axes.keys()), dtype=float)
pval_mat = pd.DataFrame(index=FP_MD, columns=list(axes.keys()), dtype=float)
delta_by_axis = {}
for ax_name, (col, lo, hi) in axes.items():
    sub = bt[bt[col].isin([lo, hi])]
    agg = sub.groupby(['target', col]).bedroc20_gbsa.mean().unstack(col)
    delta = (agg[hi] - agg[lo]).reindex(targets)
    delta_by_axis[ax_name] = delta
    for feat in FP_MD:
        x = FP[feat].values; y = delta.values
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 5: rho_mat.loc[feat, ax_name] = np.nan; pval_mat.loc[feat, ax_name] = np.nan; continue
        rho, _ = spearmanr(x[m], y[m])
        rho_mat.loc[feat, ax_name] = rho
        pval_mat.loc[feat, ax_name] = perm_p(rho, x[m], y[m], n=3000)

rho_mat = rho_mat.astype(float)
pval_mat = pval_mat.astype(float)
print('Significant (perm p < 0.05):')
sig = (pval_mat < 0.05).stack()
sig_pairs = sig[sig].index.tolist()
for feat, ax in sig_pairs:
    print(f'  {feat:32s} × {ax:22s}  ρ = {rho_mat.loc[feat, ax]:+.2f}   p = {pval_mat.loc[feat, ax]:.3f}')

# Apply Benjamini-Hochberg FDR correction across all (feature × axis) tests
from statsmodels.stats.multitest import multipletests
flat = pval_mat.stack().dropna()
_, q_flat, _, _ = multipletests(flat.values, method='fdr_bh')
qval_mat = pval_mat.copy().astype(float)
for (feat, ax), q in zip(flat.index, q_flat):
    qval_mat.loc[feat, ax] = q
print('\nFDR-adjusted (BH) significant at q < 0.10:')
qsig = (qval_mat < 0.10).stack()
for (feat, ax) in qsig[qsig].index:
    print(f'  {feat:32s} × {ax:22s}  ρ = {rho_mat.loc[feat, ax]:+.2f}   p = {pval_mat.loc[feat, ax]:.3f}   q = {qval_mat.loc[feat, ax]:.3f}')

In [ ]:

# Heatmap
fig, ax = plt.subplots(figsize=(7, max(8, 0.28*len(rho_mat))))
im = ax.imshow(rho_mat.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(rho_mat.shape[1])); ax.set_xticklabels(rho_mat.columns, rotation=25, ha='right')
ax.set_yticks(range(rho_mat.shape[0])); ax.set_yticklabels(rho_mat.index, fontsize=8)
# annotate: ρ value + * if p<0.05, ** if p<0.01
for i in range(rho_mat.shape[0]):
    for j in range(rho_mat.shape[1]):
        rho = rho_mat.values[i, j]; p = pval_mat.values[i, j]
        if not np.isfinite(rho): continue
        stars = '**' if p < 0.01 else ('*' if p < 0.05 else '')
        col = WHITE if abs(rho) > 0.55 else NAVY
        ax.text(j, i, f'{rho:+.2f}{stars}', ha='center', va='center', fontsize=7, color=col)
cbar = plt.colorbar(im, ax=ax, label='Spearman ρ  (per-target preference delta vs MD feature)', shrink=0.6)
cbar.ax.yaxis.label.set_color(NAVY)
ax.set_title('MD-signature × GBSA-parameter-preference correlations\n(** = p<0.01, * = p<0.05, permutation, 8 targets)')

In [ ]:

# Scatter panels for the three significant correlations
picks = [
    ('ligand_partial_charge_sum', 'intdiel 1→4', 'ligand net charge (e)',
     'BEDROC(intdiel=4) − BEDROC(intdiel=1)',
     'Negatively charged ligands PREFER intdiel=4 (higher internal dielectric)'),
    ('protein_formal_charge',     'intdiel 1→4', 'protein formal charge (e)',
     'BEDROC(intdiel=4) − BEDROC(intdiel=1)',
     'Negatively charged proteins PREFER intdiel=4'),
    ('ifp_tanimoto_median_vs_ref','surften 0→0.0072',
     'IFP-Tanimoto median vs ref (higher = pose keeps its fingerprint)',
     'BEDROC(surften=0.0072) − BEDROC(surften=0)',
     'Fingerprint-stable poses BENEFIT from the SA nonpolar term'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (feat, ax_name, xlab, ylab, title) in zip(axes, picks):
    x = FP[feat]
    y = delta_by_axis[ax_name]
    ax.scatter(x, y, s=90, color=GOLD, edgecolors=NAVY, linewidths=0.8, zorder=3)
    for t in x.index:
        ax.annotate(t, (x[t], y[t]), fontsize=8, color=NAVY,
                    xytext=(5, 4), textcoords='offset points')
    # regression line
    from numpy.polynomial import polynomial as P
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() >= 3:
        coef = np.polyfit(x[m], y[m], 1)
        xg = np.linspace(x[m].min(), x[m].max(), 30)
        ax.plot(xg, np.polyval(coef, xg), color=NAVY, ls='--', lw=1, alpha=0.7)
    ρ, _ = spearmanr(x[m], y[m])
    ax.axhline(0, color=GREY, ls=':', lw=1)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.set_axisbelow(True); ax.grid(True, color=GREY, alpha=0.4)
    ax.set_title(f'{title}\nSpearman ρ = {ρ:+.2f}', fontsize=9)
plt.tight_layout()

**Interpretation — honest read of the correlations:**

Pitfall caught in review: `ligand_partial_charge_sum` is essentially numerical noise here. All ligands are constructed net-neutral, so the values are float-rounding artefacts of order 1e-5. Spearman rank on noise produces spurious "significant" p-values (rank of noise is stable but arbitrary). We dropped `ligand_partial_charge_sum` from this section — any correlation involving it was an artefact.

With 24 MD features × 4 GBSA axes = 96 tests, expect ≈5 spurious p<0.05 findings by chance. We apply BH-FDR correction (q<0.10) to identify real signals.

**Result.** No MD × GBSA-parameter correlation survives BH-FDR at q<0.10. The stdout above prints `FDR-adjusted (BH) significant at q < 0.10:` followed by an empty list; the best q across all 96 tests is q ≈ 0.132 (NB 27 adds ligand-chem features and reports the same). An earlier version of this notebook stated "Both survive BH-FDR at q<0.10" — that was a bug in the interpretation cell, not in the code. Retracted.

Top-ranked associations below are descriptive only. Physical narratives are hypotheses for future validation, not evidence.

- **`protein_formal_charge` × `intdiel` 1→4** (ρ ≈ −0.83, uncorrected p ≈ 0.018, BH q ≈ 0.44).
   Hypothesis (unconfirmed): more-negative proteins may benefit from intdiel=4 because a higher internal dielectric handles distributed protein charges better. Not significant after correction.

- **`ifp_tanimoto_median_vs_ref` × `surften` 0→0.0072** (ρ ≈ +0.79, uncorrected p ≈ 0.029, BH q > 0.4).
   Hypothesis (unconfirmed): ligands that keep their initial interaction fingerprint (stable poses) may benefit from the SA nonpolar term because a rigid pose has a well-defined buried nonpolar surface. Not significant after correction.

**What we still cannot say:**
- **Per-complex ligand chemistry** (net charge, LogP, aromatic-ring count, HBA/HBD): the topology-only features here are net-neutral by construction, so ligand charge is not meaningful. NB 27 revisits this with proper RDKit descriptors from `system.top + system.gro`.
- **Per-target GBSA-parameter selection rules**: the two candidates above are hypotheses. Do not deploy.

**Practical:**
- No per-target GBSA-parameter rule is supported by this data. GBSA-locked (`igb2_di4_salt0.15_st0.0072`) is what the paper deploys, and this notebook doesn't identify a data-driven basis for varying it per target.
- The two candidate rules (intdiel from protein charge; surften from IFP-Tanimoto) are directions for future validation on the 18-target locked-parameter set.
- `igb` and `saltcon` show no correlations near significance either before or after correction.

In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
